In [ ]:
'''
Aharonov, et al - A Polynomial Quantum Algorithm for Approximating the Jones Polynomial.pdf
'''

import numpy as np
from qiskit import QuantumRegister, QuantumCircuit

# Hopf 3-chain. Input: "4 4 2 2"=\sigma_4\sigma_4\sigma_2\sigma_2 in B6, k=5 read left-to-right

strand_count = 6
k = 5

# Pre-compute: \lambda_1,...,\lambda_5

lam = [0] + [np.sin(np.pi*v/k) for v in range(1,k)] + [0]

A = (0+1j)*np.exp((0-1j)*np.pi/10)

# Pre-compute: \Phi_1,...,\Phi_5

phi = [A*np.array([[0,0,0,0],
                    [0,lam[v+1]/lam[v],np.sqrt(lam[v+1]*lam[v-1])/lam[v],0],
                    [0,np.sqrt(lam[v+1]*lam[v-1])/lam[v],lam[v-1]/lam[v],0],
                    [0,0,0,0]]) + A**(-1)*np.identity(4) for v in range(1,k)]
phi_inv = [A**(-1)*np.array([[0,0,0,0],
                    [0,lam[v+1]/lam[v],np.sqrt(lam[v+1]*lam[v-1])/lam[v],0],
                    [0,np.sqrt(lam[v+1]*lam[v-1])/lam[v],lam[v-1]/lam[v],0],
                    [0,0,0,0]]) + A*np.identity(4) for v in range(1,k)]
from qiskit.quantum_info import Operator
# Pre-compute size of ancilla and counter

anc_size = int(np.ceil(np.log2(2*k)))

# Create phiGates which is a set of gates that executes phi with anc_size-many control qubits

phiOperators = [Operator(U) for U in phi]
phiGates = []
for i in range(len(phi)):
    phiCirc = QuantumCircuit(2)
    phiCirc.unitary(phi[i],[0,1])
    phiGates.append(phiCirc.to_gate(label=f"phi_{i}"))

ctrl_phiGates = [U.control(anc_size) for U in phiGates]

phiOperators_inv = [Operator(U) for U in phi_inv]
phiGates_inv = []
for i in range(len(phi_inv)):
    phiCirc = QuantumCircuit(2)
    phiCirc.unitary(phi_inv[i],[0,1])
    phiGates_inv.append(phiCirc.to_gate(label=f"phi_inv_{i}"))

ctrl_phiGates_inv = [U.control(anc_size) for U in phiGates_inv]

# Quantum registers

path = QuantumRegister(strand_count,'path')
anc = QuantumRegister(anc_size, 'anc')
const = QuantumRegister(anc_size,'const')

# Create quantum circuit

qc = QuantumCircuit(path, anc, const)

# Creating walk

from qiskit.circuit.library import ModularAdderGate

adder = ModularAdderGate(len(anc))
ctrl_adder = adder.control(1)

def create_walkGate(anc_size):
    control_reg = QuantumRegister(1,'path')
    anc_reg = QuantumRegister(anc_size, 'anc')
    const_reg = QuantumRegister(anc_size,'const')
    circ = QuantumCircuit(control_reg, anc_reg, const_reg)

    circ.x(const_reg[1:anc_size])
    circ.append(ctrl_adder, [control_reg] + list(const_reg) + list(anc_reg))
    circ.x(const_reg[1:anc_size])

    circ.x(control_reg)
    circ.append(ctrl_adder, [control_reg] + list(const_reg) + list(anc_reg))
    circ.x(control_reg)
    return circ.to_gate()

def walk(circ, path_reg, anc_reg, const_reg, walker, steps):
    for i in range(strand_count - steps, strand_count):
        circ.append(walker, [path_reg[i]] + list(anc_reg) + list(const_reg))

def unwalk(circ, path_reg, anc_reg, const_reg, walker, steps):
    for i in reversed(range(strand_count - steps, strand_count)):
        circ.append(walker.inverse(), [path_reg[i]] + list(anc_reg) + list(const_reg))

# Creating a function that executes sigma_i
walkGate = create_walkGate(len(anc))
def sigma_i(circ, path_reg, anc_reg, const_reg, generator):
    '''
    circ: a quantum circuit built with path, ancilla, and constant register
    anc_size: size of ancilla circuit
    generator: a non-zero integer denoting the i-th braid generator. Negative integer denotes inverse generator
    '''
    
    walk(circ, path_reg, anc_reg, const_reg, walkGate, abs(generator) - 1)

    # Multiplex. The following will execute the correct Phi when anc = v
    if generator > 0:
        for v in range(1, k):
            zero_bits = [i for i in range(len(anc_reg)) if not (v >> i) & 1] # For each vertex v, find the index of 0s in a binary expansion of v
            
            circ.x(anc_reg[zero_bits]) # Flip all the qubits in zero_bits to 1. This way, the multi-controlled phiGates only activates when the ancilla is precisely v
            circ.append(ctrl_phiGates[v-1], list(anc_reg) + [path_reg[generator - 1], path_reg[generator]]) # Execute Phi_v on the i-th and (i+1)-th qubit with ancilla as controls
            circ.x(anc_reg[zero_bits]) # Undo the flip to prepare for the next v
            
        unwalk(circ, path_reg, anc_reg, const_reg, walkGate, generator - 1)
    elif generator < 0:
        for v in range(1, k):
            zero_bits = [i for i in range(len(anc_reg)) if not (v >> i) & 1] # For each vertex v, find the index of 0s in a binary expansion of v
            
            circ.x(anc_reg[zero_bits]) # Flip all the qubits in zero_bits to 1. This way, the multi-controlled phiGates only activates when the ancilla is precisely v
            circ.append(ctrl_phiGates_inv[v-1], list(anc_reg) + [path_reg[abs(generator) - 1], path_reg[abs(generator)]]) # Execute Phi_v on the i-th and (i+1)-th qubit with ancilla as controls
            circ.x(anc_reg[zero_bits]) # Undo the flip to prepare for the next v
            
        unwalk(circ, path_reg, anc_reg, const_reg, walkGate, abs(generator) - 1)
    else:
        return "Error: 0-th generator does not exists"

def braid(circ, path_reg, anc_reg, const_reg, sequence):
    '''
    sequence: left-to-right sequence of braids
    '''
    for i in sequence:
        sigma_i(circ, path_reg, anc_reg, const_reg, i)


In [27]:
from qiskit import ClassicalRegister
braidCirc = QuantumCircuit(path, anc, const)
braid(braidCirc, path, anc, const, [4, 4, 2, 2])
braidGate = braidCirc.to_gate()
ctrl_braidGate = braidGate.control(1)

Re_had = QuantumRegister(1, 'Re_had')
Re_meas = ClassicalRegister(1, 'Re_meas')
Re_had_circ = QuantumCircuit(Re_had, path, anc, const, Re_meas)

# Initializations of: path to 1010...10, ancilla to 00...01, and const to 00...01

Re_had_circ.x(path[[i for i in range(strand_count) if i % 2  == 0]])
Re_had_circ.x(anc[0])
Re_had_circ.x(const[0])
# Initialization of Re_had to |0>+|1>
Re_had_circ.h(Re_had[0])
Re_had_circ.append(ctrl_braidGate, [Re_had[0]] + list(path) + list(anc) + list(const))
Re_had_circ.h(Re_had[0])

Re_had_circ.measure(Re_had[0], Re_meas[0])

In [28]:
Im_had = QuantumRegister(1, 'Im_had')
Im_meas = ClassicalRegister(1, 'Im_meas')
Im_had_circ = QuantumCircuit(Im_had, path, anc, const, Im_meas)

# Initializations of: path to 1010...10, ancilla to 00...01, and const to 00...01

Im_had_circ.x(path[[i for i in range(strand_count) if i % 2  == 0]])
Im_had_circ.x(anc[0])
Im_had_circ.x(const[0])

# Initialization of Re_had to |0>+|1>
Im_had_circ.h(Im_had[0])
Im_had_circ.sdg(Im_had[0])
Im_had_circ.append(ctrl_braidGate, [Im_had[0]] + list(path) + list(anc) + list(const))
Im_had_circ.h(Im_had[0])

Im_had_circ.measure(Im_had[0], Im_meas[0])

In [30]:
from qiskit import transpile
from qiskit_aer import AerSimulator

backend = AerSimulator()
Re_had_circ_comp = transpile(Im_had_circ, backend)
Re_job = backend.run(Re_had_circ_comp, shots=100000)
Re_result = Re_job.result()

print(Re_result.get_counts(Re_had_circ_comp))

{'1': 49938, '0': 50062}
